# **Data Parsing and Processing for Json Line files**

### Data Parsing for Playbooks 

In [25]:
import json
from langchain_core.documents import Document

### Data Overview

In [11]:
# Open the file and read the first line
with open("../data/playbooks/incident_response_playbook_dataset.jsonl", 'r') as f:
    first_line = f.readline()
    data = json.loads(first_line)
    
# Having an overview of the dataset
print("Keys in this record:", data.keys())
print("\nFull first record:")
print(json.dumps(data, indent=2))

Keys in this record: dict_keys(['incident_id', 'incident_type', 'target_asset', 'detection_source', 'initial_vector', 'tactics_techniques', 'severity', 'playbook_steps', 'final_status', 'response_duration_total_min', 'tags'])

Full first record:
{
  "incident_id": "IR-2025-0012",
  "incident_type": "Ransomware",
  "target_asset": "Windows AD Server",
  "detection_source": "EDR Alert - Suspicious File Encryption",
  "initial_vector": "Email Attachment (Phishing)",
  "tactics_techniques": [
    {
      "tactic": "Initial Access",
      "technique": "Phishing"
    },
    {
      "tactic": "Execution",
      "technique": "User Execution"
    },
    {
      "tactic": "Impact",
      "technique": "Data Encrypted for Impact"
    }
  ],
  "severity": "High",
  "playbook_steps": [
    {
      "phase": "Identification",
      "action": "Triage alert, confirm IOC via EDR, snapshot affected host",
      "tools": [
        "CrowdStrike Falcon",
        "Velociraptor"
      ],
      "response_time_m

## Manual Line by Line Parsing and Processing 

In [12]:
import json
from langchain_core.documents import Document

# the exact path to the file stored in the variable "Filepaht"
file_path = "../data/playbooks/cleaned_incident_response_playbook_dataset.jsonl"

# Creating an empty list to hold all documents
playbook_documents = []

# Read the file line by line
with open(file_path, 'r') as f: #reads the file from the filepath using the enconding UTF-8
    for line in f:  #Iterates over each line 
        obj = json.loads(line)
        
        # Building the page_content from the playbook steps
        steps_text = ""
        for step in obj['playbook_steps']:
            steps_text += f"Phase {step['phase']}: {step['action']}\n"
        
        # Building metadata (extra info)
        metadata = {
            "incident_id": obj['incident_id'],
            "incident_type": obj['incident_type'],
            "severity": obj['severity'],
            "final_status": obj['final_status']
        }
        
        #Creating a Document
        doc = Document(
            page_content=steps_text,
            metadata=metadata
        )
        
        #Adding the document to the list
        playbook_documents.append(doc)

# Checking what has been loaded
print(f"Loaded {len(playbook_documents)} documents")
print(f"First document page_content:\n{playbook_documents[0].page_content}")
print(f"First document metadata: {playbook_documents[0].metadata}")

Loaded 174 documents
First document page_content:
Phase Identification: Triage alert, confirm IOC via EDR, snapshot affected host
Phase Containment: Isolate host from network, disable compromised user account
Phase Eradication: Remove malicious binaries, patch vulnerabilities
Phase Recovery: Restore from backups, monitor for reinfection
Phase Lessons Learned: Conduct IR debrief, update detection rules, user awareness training

First document metadata: {'incident_id': 'IR-2025-0012', 'incident_type': 'Ransomware', 'severity': 'High', 'final_status': 'Resolved'}


#### Finding and fixing the error in the dataset

In [13]:
file_path = "../data/playbooks/incident_response_playbook_dataset.jsonl"

with open(file_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()
    
bad_line = lines[104]  # line 105 is index 104
print(f"Bad line (length {len(bad_line)} chars):")
print(bad_line)
print("\nCharacter 296:", repr(bad_line[296]))

Bad line (length 1133 chars):
{"incident_id":"IR-2025-0116","incident_type":"Ransomware","target_asset":"Virtualized Infrastructure","detection_source":"EDR Alert - File Encryption","initial_vector":"VM Escape Exploit","tactics_techniques":[{"tactic":"Initial Access","technique":"Exploit Public-Facing Application"},{"tactic","Impact","technique":"Data Encrypted for Impact"}],"severity":"Critical","playbook_steps":[{"phase":"Identification","action":"Triage alert, snapshot VMs","tools":["SentinelOne","VMware vSphere"],"response_time_min":15},{"phase":"Containment","action":"Isolate VMs, block C2","tools":["EDR","NSX Firewall"],"response_time_min":10},{"phase":"Eradication","action":"Remove ransomware, patch hypervisor","tools":["YARA","Patching Platform"],"response_time_min":60},{"phase":"Recovery","action":"Restore VMs from backups, monitor for reinfection","tools":["Veeam","SIEM"],"response_time_min":120},{"phase":"Lessons Learned","action":"Review VM security, update policies","tools

- Observation:

Found the issue, the second object for the `tactic-technique` has wrong format `"tactic","Impact",` instead of `"tactic":"Impact",`

In [14]:
# regex library
import re

#Specifying file paths
file_path = "../data/playbooks/incident_response_playbook_dataset.jsonl"
output_path = "../data/playbooks/cleaned_incident_response_playbook_dataset.jsonl"

# Empty variables to store the cleaned lines and bad lines
cleaned_lines = []
bad_lines = []

with open(file_path, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f, start=1):
        line = line.strip()
        if not line:
            continue
            
        # Fix the specific error: {"tactic","Impact",...} -> {"tactic":"Impact",...}
        line = re.sub(r'\{"tactic","([^"]+)"', r'{"tactic":"\1"', line)
        
        # Validate if the line is now valid JSON
        try:
            json.loads(line)
            cleaned_lines.append(line)
        except json.JSONDecodeError as e:
            bad_lines.append((i, str(e)))

# Save cleaned lines
with open(output_path, 'w', encoding='utf-8') as f:
    for line in cleaned_lines:
        f.write(line + '\n')

print(f"Cleaned {len(cleaned_lines)} lines")
print(f"Skipped {len(bad_lines)} lines")
if bad_lines:
    print("First skipped line:", bad_lines[0])

Cleaned 174 lines
Skipped 0 lines


## Data Parsing for Log Files

In [26]:
## Importing Libraries
import pandas as pd

In [27]:
# Looking at the dataset to see which is neeeded for this project
# specifying file path
file_path = '../data/logs/Subset_precinct6_21M Main/signals/signals.parquet' 

# Loading only the first 1000 rows to get a feel for the data
try:
    df = pd.read_parquet(file_path)
    print(f"Successfully loaded {len(df)} rows.")
    print("\n Column Names")
    print(df.columns.tolist())
    print("\n First 2 Rows (Preview)")
    print(df.head(2))
except FileNotFoundError:
    print(f"Error: File not found at '{file_path}'. Please check the path.")
except Exception as e:
    print(f"An error occurred: {e}")

Successfully loaded 2100363 rows.

 Column Names
['timestamp', 'message_type', 'stream_name', 'pipeline', 'src_ip', 'dst_ip', 'src_port', 'dst_port', 'protocol', 'src_host', 'dst_host', 'username', 'action', 'severity', 'vendor_code', 'message_sanitized', 'label_binary', 'label_confidence', 'attack_techniques', 'attack_tactics', 'defense_techniques', 'mo_name', 'suspicion_score', 'lifecycle_stage', 'disposition', 'disposition_category', 'is_false_positive', 'status_name', 'incident_ids', 'matched_rules', 'set_roles', 'product_name', 'vendor_name']

 First 2 Rows (Preview)
      timestamp  message_type    stream_name pipeline         src_ip  \
0 -2.111912e+10  GEO_IP_BLOCK  barracuda_waf   syslog  100.64.95.151   
1 -2.111912e+10  GEO_IP_BLOCK  barracuda_waf   syslog  100.64.95.151   

           dst_ip src_port dst_port protocol       src_host  ...  \
0  10.184.211.180    47220      443      NaN  10.149.108.24  ...   
1  10.184.211.180    47220      443      NaN  10.149.108.24  ...   


In [28]:
#Loading the dataset
df = pd.read_parquet("../data/logs/Subset_precinct6_21M Main/signals/signals.parquet")  

print(f"Total rows in full dataset: {len(df):,}")


Total rows in full dataset: 2,100,363


In [29]:
# using a sample from the dataset
sample_df = df.sample(n=5000, random_state=42).copy() 

print(f"Sampled {len(sample_df):,} rows for prototyping.")

Sampled 5,000 rows for prototyping.


- Note

Had to return back to modify the sample from 50,000 to 5,000 to enable my CPU process handle embedding proficiently.

In [30]:
# Keep only rows that have a valid 'message_sanitized' column.
sample_df = sample_df.dropna(subset=['message_sanitized'])

# Convert the message to string 
sample_df['message_sanitized'] = sample_df['message_sanitized'].astype(str)

print(f"Cleaned sample: {len(sample_df):,} rows with text content.")

Cleaned sample: 5,000 rows with text content.


In [31]:
# Converting each row to a document
log_documents = []

for idx, row in sample_df.iterrows():
    # --- Extract the main text (page_content) ---
    page_content = row.get('message_sanitized', '').strip()
    
    # Skip if empty (after stripping)
    if not page_content:
        continue

    # Building metadata (keep only the most useful fields) 
    # This keeps the vector store lightweight.
    metadata = {
        'timestamp': str(row.get('timestamp', '')),
        'src_ip': row.get('src_ip', ''),
        'dst_ip': row.get('dst_ip', ''),
        'username': row.get('username', ''),
        'severity': row.get('severity', ''),
        'label_binary': row.get('label_binary', ''),
        'src_port': row.get('src_port', ''),
        'dst_port': row.get('dst_port', ''),
        'protocol': row.get('protocol', ''),
    }
    
    # Optional: Adding lists (like attack_tactics) as a comma-separated string
    tactics = row.get('attack_tactics', [])
    if isinstance(tactics, list):
        metadata['attack_tactics'] = ', '.join(tactics)
    else:
        metadata['attack_tactics'] = str(tactics)
        
# Creating the LangChain Document
    doc = Document(
        page_content=page_content,
        metadata=metadata
    )
    log_documents.append(doc)

In [32]:
#Checking the result
print(f"\n Successfully created {len(log_documents):,} Document objects.")
print(f"\n First Document Preview")
print(f"Page Content (first 300 chars):\n{log_documents[0].page_content[:300]}...")
print(f"\nMetadata Keys: {log_documents[0].metadata.keys()}")
print(f"Metadata Sample: {log_documents[0].metadata}")


 Successfully created 4,120 Document objects.

 First Document Preview
Page Content (first 300 chars):
<Event xmlns='http://schemas.microsoft.com/win/2004/08/events/event'><ORG-0111><Provider Name='Microsoft-ORG-0362-Security-Auditing' Guid='{54849625-5478-4994-a5ba-3e3b0328c30d}'/><EventID>4634</EventID><Version>0</Version><Level>0</Level><Task>12545</Task><Opcode>0</Opcode><Keywords>0x8020000000000...

Metadata Keys: dict_keys(['timestamp', 'src_ip', 'dst_ip', 'username', 'severity', 'label_binary', 'src_port', 'dst_port', 'protocol', 'attack_tactics'])
Metadata Sample: {'timestamp': '1721994830.9275465', 'src_ip': '', 'dst_ip': '', 'username': 'USER-1691', 'severity': '', 'label_binary': 'benign', 'src_port': '', 'dst_port': '', 'protocol': '', 'attack_tactics': '[]'}


### **Saving the datasets to use in other Notebooks**

In [22]:
#Making a directory 
import os
os.makedirs("../data/processed", exist_ok=True)

In [33]:
#Creating the log_document processed file
import pickle
with open('../data/processed/log_documents.pkl', 'wb') as f:
    pickle.dump(log_documents, f)

In [24]:
#Saving the processed playbook dataset
with open('../data/processed/playbook_documents.pkl', 'wb') as f:
    pickle.dump(playbook_documents, f)